# GodelReplay: PermutedMNIST Benchmark
### GodelPlugin (Fisher-scaled EWC-DR) + Avalanche Replay -- Two-Layer Continual Learning

> **Hypothesis:** GodelReplay (Replay + GodelPlugin) achieves lower catastrophic forgetting
> than either Replay-only or EWC-only on PermutedMNIST (10 tasks).

| Strategy | Mechanism | Expected Forgetting |
|----------|-----------|---------------------|
| Naive | None (sanity baseline) | ~90% |
| Replay-only | Past-task sample buffer (mem=500) | ~8-12% |
| EWC-only (GodelPlugin) | Fisher-scaled regularization | ~31.5% (reproduces prior result) |
| **GodelReplay** | **Replay + GodelPlugin** | **< Replay-only (target)** |

**Part of the Two-Layer GodelAI Architecture:**
```
Training-time  ->  GodelAI / GodelReplay  : Fisher Scaling + EWC-DR + Replay
Inference-time ->  GodelAI-Lite           : MemPalace + MACP + GIFP
Together       ->  Complete memory system for continual AI
```

*C-S-P: Compression -> State -> Propagation | YSenseAI Ecosystem 2026*

## 1. Install Dependencies

In [1]:
import subprocess, sys

def _install(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

_install('avalanche-lib', 'torch', 'torchvision')
print('avalanche-lib installed.')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.2/134.2 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 993.4/993.4 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 585.2/585.2 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.9/101.9 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.5/869.5 kB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.1/222.1 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.4/172.4 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 534.6/534.6 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.2/165.2 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 2. Load GodelAI Repository

In [2]:
import subprocess, os, sys

GODELAI_DIR = '/kaggle/working/godelai-repo'

if not os.path.exists(GODELAI_DIR):
    print('Cloning creator35lwb-web/godelai...')
    result = subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/creator35lwb-web/godelai.git', GODELAI_DIR],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print('Clone error:', result.stderr)
        raise RuntimeError('Failed to clone godelai repo. Ensure internet is enabled.')
    print('Cloned successfully.')
else:
    print('godelai-repo already present.')

if GODELAI_DIR not in sys.path:
    sys.path.insert(0, GODELAI_DIR)

from godelai.avalanche_plugin import GodelPlugin
from godelai.strategies.godel_replay import create_godel_replay_strategy
print('GodelPlugin and GodelReplay strategy imported successfully.')


Cloning creator35lwb-web/godelai...
Cloned successfully.


2026-04-28 18:11:38.055627: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777399898.249027      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777399898.302994      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777399898.739974      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777399898.740017      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777399898.740020      23 computation_placer.cc:177] computation placer alr

GodelPlugin and GodelReplay strategy imported successfully.


## 3. Experiment Configuration

In [3]:
import torch

def _resolve_device():
    # P100 is sm_60 -- PyTorch >=2.0 requires sm_70+. Fall back to CPU.
    if not torch.cuda.is_available():
        return 'cpu'
    major, minor = torch.cuda.get_device_capability(0)
    if major >= 7:
        return 'cuda'
    name = torch.cuda.get_device_name(0)
    print('[Warning] GPU sm_' + str(major) + str(minor) + ' (' + name + ') < sm_70 -- falling back to CPU.')
    return 'cpu'

CONFIG = {
    'n_experiences': 10,
    'seed': 42,
    'train_epochs': 5,
    'train_mb_size': 128,
    'eval_mb_size': 256,
    'lr': 0.001,
    'device': _resolve_device(),
    'ewc_lambda': 400.0,
    'fisher_scaling': 'global_max',
    'propagation_gamma': 2.0,
    't_score_window': 50,
    'mem_size': 500,
}

print('Configuration:')
for k, v in CONFIG.items():
    print('  ' + str(k).ljust(22) + ': ' + str(v))
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print('GPU: ' + torch.cuda.get_device_name(0) + ' | sm_' + str(cap[0]) + str(cap[1]) + ' | VRAM: ' + str(round(vram, 1)) + ' GB')
print('Effective device: ' + CONFIG['device'])


[Warning] GPU sm_60 (Tesla P100-PCIE-16GB) < sm_70 -- falling back to CPU.
Configuration:
  n_experiences         : 10
  seed                  : 42
  train_epochs          : 5
  train_mb_size         : 128
  eval_mb_size          : 256
  lr                    : 0.001
  device                : cpu
  ewc_lambda            : 400.0
  fisher_scaling        : global_max
  propagation_gamma     : 2.0
  t_score_window        : 50
  mem_size              : 500
GPU: Tesla P100-PCIE-16GB | sm_60 | VRAM: 17.1 GB
Effective device: cpu


/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 Tesla P100-PCIE-16GB which is of cuda capability 6.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
Tesla P100-PCIE-16GB with CUDA capability sm_60 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to use the Tesla P100-PCIE-16GB GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  queued_call()


## 4. Run -- 4-Strategy Comparison
> Each strategy trains on 10 PermutedMNIST tasks sequentially.
> After each task, all prior test streams are evaluated.
> Estimated runtime: 40-90 min on CPU (P100 sm_60 forces CPU fallback).

In [4]:
import sys
sys.path.insert(0, GODELAI_DIR)

from experiments.permutedmnist_godelreplay import run_strategy, CONFIG

strategies = ['naive', 'replay_only', 'ewc_only', 'godel_replay']
results = []

for name in strategies:
    r = run_strategy(name, CONFIG)
    results.append(r)
    acc = r['final_accuracy']
    forg = r['avg_forgetting']
    print('  -> ' + name + ': acc=' + str(acc) + ', forgetting=' + str(forg))

print('All strategies complete.')


[Warning] GPU sm_60 < sm_70 (Tesla P100-PCIE-16GB) — falling back to CPU (P100 known incompatibility with current PyTorch).

  Strategy: NAIVE


100%|██████████| 9.91M/9.91M [00:00<00:00, 17.9MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 482kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.45MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.39MB/s]



  Task 1/10
-- >> Start of training phase << --
-- >> Start of training phase << --
100%|██████████| 469/469 [00:12<00:00, 37.31it/s]
Epoch 0 ended.
	Loss_Epoch/train_phase/train_stream/Task000 = 0.2523
	Top1_Acc_Epoch/train_phase/train_stream/Task000 = 0.9241
100%|██████████| 469/469 [00:12<00:00, 37.31it/s]
Epoch 0 ended.
	Loss_Epoch/train_phase/train_stream/Task000 = 0.2523
	Top1_Acc_Epoch/train_phase/train_stream/Task000 = 0.9241
100%|██████████| 469/469 [00:12<00:00, 37.65it/s]
Epoch 1 ended.
	Loss_Epoch/train_phase/train_stream/Task000 = 0.0965
	Top1_Acc_Epoch/train_phase/train_stream/Task000 = 0.9700
100%|██████████| 469/469 [00:12<00:00, 37.65it/s]
Epoch 1 ended.
	Loss_Epoch/train_phase/train_stream/Task000 = 0.0965
	Top1_Acc_Epoch/train_phase/train_stream/Task000 = 0.9700
100%|██████████| 469/469 [00:12<00:00, 38.73it/s]
Epoch 2 ended.
	Loss_Epoch/train_phase/train_stream/Task000 = 0.0637
	Top1_Acc_Epoch/train_phase/train_stream/Task000 = 0.9806
100%|██████████| 469/469 [00:1

/usr/local/lib/python3.12/dist-packages/avalanche/training/plugins/replay.py:123: DeprecationWarning: Call to deprecated function update (removal in version 0.7: switch to pre_adapt and post_adapt)
  self.storage_policy.update(strategy, **kwargs)



Epoch 4 ended.
	Loss_Epoch/train_phase/train_stream/Task000 = 0.0386
	Top1_Acc_Epoch/train_phase/train_stream/Task000 = 0.9872
-- >> End of training phase << --
-- >> End of training phase << --
-- >> Start of eval phase << --
-- >> Start of eval phase << --
-- Starting eval on experience 0 (Task 0) from test stream --
0it [00:00, ?it/s]

...

-- Starting eval on experience 0 (Task 0) from test stream --
100%|██████████| 40/40 [00:01<00:00, 23.98it/s]
> Eval on experience 0 (Task 0) from test stream ended.
	ExperienceForgetting/eval_phase/test_stream/Task000/Exp000 = 0.1787
	Loss_Exp/eval_phase/test_stream/Task000/Exp000 = 1.5875
	Top1_Acc_Exp/eval_phase/test_stream/Task000/Exp000 = 0.7974
100%|██████████| 40/40 [00:01<00:00, 23.99it/s]
> Eval on experience 0 (Task 0) from test stream ended.
	Loss_Exp/eval_phase/test_stream/Task000/Exp000 = 1.5875
	Top1_Acc_Exp/eval_phase/test_stream/Task000/Exp000 = 0.7974
-- Starting eval on experience 1 (Task 0) from test stream --
0it [00:00, ?it/s]-- Starting eval on experience 1 (Task 0) from test stream --
100%|██████████| 40/40 [00:01<00:00, 24.26it/s]
> Eval on experience 1 (Task 0) from test stream ended.
	ExperienceForgetting/eval_phase/test_stream/Task000/Exp001 = 0.1709
	Loss_Exp/eval_phase/test_stream/Task000/Exp001 = 1.2868
	Top1_Acc_Exp/eval_phase/test_stream/Task000/Exp001 =

/usr/local/lib/python3.12/dist-packages/avalanche/training/plugins/replay.py:123: DeprecationWarning: Call to deprecated function update (removal in version 0.7: switch to pre_adapt and post_adapt)
  self.storage_policy.update(strategy, **kwargs)



Epoch 4 ended.
	Loss_Epoch/train_phase/train_stream/Task000 = 0.0177
	Top1_Acc_Epoch/train_phase/train_stream/Task000 = 0.9946
[GodelPlugin] Fisher Scaling: max raw=0.004538, max scaled=1.000000, scale_problem=False
[GodelPlugin] EWC-DR consolidated. Dead params: 99.9%
[GodelPlugin] Experience 10 complete. Avg T-Score: 0.9958, Sleep events: 0
-- >> End of training phase << --
-- >> End of training phase << --
-- >> Start of eval phase << --
-- >> Start of eval phase << --
-- Starting eval on experience 0 (Task 0) from test stream --
0it [00:00, ?it/s]

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


-- Starting eval on experience 0 (Task 0) from test stream --
100%|██████████| 40/40 [00:01<00:00, 23.16it/s]
> Eval on experience 0 (Task 0) from test stream ended.
	ExperienceForgetting/eval_phase/test_stream/Task000/Exp000 = 0.2399
	Loss_Exp/eval_phase/test_stream/Task000/Exp000 = 2.4534
	Top1_Acc_Exp/eval_phase/test_stream/Task000/Exp000 = 0.7362
100%|██████████| 40/40 [00:01<00:00, 23.16it/s]
> Eval on experience 0 (Task 0) from test stream ended.
	Loss_Exp/eval_phase/test_stream/Task000/Exp000 = 2.4534
	Top1_Acc_Exp/eval_phase/test_stream/Task000/Exp000 = 0.7362
-- Starting eval on experience 1 (Task 0) from test stream --
0it [00:00, ?it/s]-- Starting eval on experience 1 (Task 0) from test stream --
100%|██████████| 40/40 [00:01<00:00, 23.94it/s]
> Eval on experience 1 (Task 0) from test stream ended.
	ExperienceForgetting/eval_phase/test_stream/Task000/Exp001 = 0.2025
	Loss_Exp/eval_phase/test_stream/Task000/Exp001 = 1.7229
	Top1_Acc_Exp/eval_phase/test_stream/Task000/Exp001 =

## 5. Results

In [5]:
print('\n' + '='*70)
print('  GODELREPLAY RESULTS -- PermutedMNIST (10 tasks, seed=42)')
print('='*70)
print('  ' + 'Strategy'.ljust(20) + 'Final Acc'.rjust(12) + 'Avg Forgetting'.rjust(16))
print('  ' + '-'*48)

for r in results:
    acc  = '{:.4f}'.format(r['final_accuracy'])  if r['final_accuracy']  is not None else 'N/A'
    forg = '{:.4f}'.format(r['avg_forgetting'])  if r['avg_forgetting']  is not None else 'N/A'
    print('  ' + r['strategy'].ljust(20) + acc.rjust(12) + forg.rjust(16))

print('='*70)

replay_forg      = next((r['avg_forgetting'] for r in results if r['strategy'] == 'replay_only'),  None)
godelreplay_forg = next((r['avg_forgetting'] for r in results if r['strategy'] == 'godel_replay'), None)
ewc_forg         = next((r['avg_forgetting'] for r in results if r['strategy'] == 'ewc_only'),     None)

if replay_forg and godelreplay_forg and replay_forg > 0:
    delta_pct = (replay_forg - godelreplay_forg) / replay_forg * 100
    verdict = 'HYPOTHESIS CONFIRMED' if godelreplay_forg < replay_forg else 'HYPOTHESIS REJECTED'
    print('  GodelReplay vs Replay-only : {:.1f}% forgetting reduction'.format(delta_pct))
    if ewc_forg:
        print('  GodelReplay vs EWC-only    : {:.1f}% forgetting'.format((ewc_forg - godelreplay_forg) / ewc_forg * 100))
    print('  Verdict: ' + verdict)



  GODELREPLAY RESULTS -- PermutedMNIST (10 tasks, seed=42)
  Strategy               Final Acc  Avg Forgetting
  ------------------------------------------------
  naive                     0.4362          0.6003
  replay_only               0.8416          0.1500
  ewc_only                  0.4999          0.5283
  godel_replay              0.8418          0.1487
  GodelReplay vs Replay-only : 0.9% forgetting reduction
  GodelReplay vs EWC-only    : 71.9% forgetting
  Verdict: HYPOTHESIS CONFIRMED


## 6. Ecosystem Connection

GodelAI operates at two validated layers:

| Layer | System | Mechanism | Result |
|-------|--------|-----------|--------|
| **Training-time** | GodelReplay | Replay + Fisher-scaled EWC-DR | PermutedMNIST result above |
| **Inference-time** | GodelAI-Lite | MemPalace + MACP + GIFP | +31.2% overall, 3/3 memory retention |

**C-S-P maps identically across both layers:**

| C-S-P Stage | Training (GodelReplay) | Inference (GodelAI-Lite) |
|-------------|----------------------|------------------------|
| Compression | Fisher Information Matrix | extract_facts() |
| State | EWC-DR penalty + old params | godelai_memory.json |
| Propagation | Replay buffer samples | Portable JSON across models |

> *"Intelligence can scale through memory, not just parameters."*

---
- [GodelAI Framework -- Zenodo DOI 10.5281/zenodo.18048374](https://zenodo.org/records/18048374)
- [GodelAI GitHub](https://github.com/creator35lwb-web/godelai)
- [GodelAI-Lite Kaggle Notebook](https://www.kaggle.com/code/creator35lwb/godelai-lite-memory-for-gemma-4)

*creator35lwb | FLYWHEEL TEAM | MACP v2.3.1*